# Neural CFR+ snapshot suffix / average-reset diagnostic on the 18-claim game

This notebook tests whether the CFR+ average strategy is being held back by stale early iterations.

It runs one six-hour measured neural CFR+ training job on the 18-claim spec and maintains several dense averages of generated current strategies:

- a normal global average from minute 0;
- fixed suffix averages starting at selected burn-in times;
- rolling epoch averages that reset every 30, 60, or 120 minutes;
- several weighting variants.

Important implementation detail: compiling the current neural CFR+ strategy to a dense policy every iteration is too expensive. Instead, this notebook observes the dense current strategy every few measured training minutes and weights each observation by the number of CFR+ iterations since the previous observation. The resulting averages are exact dense averages of the snapshot policies, not every single iteration policy.

Exact exploitability is computed periodically. The core question is whether a recent/suffix dense average beats the normal global dense average. If it does, average inertia is a real issue and the 69-claim runs should use reset/suffix/epoch averaging.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import os
import pickle
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('Could not find repo root containing liars_poker/')
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.algo.deep_cfr_diagnostics import ExactDenseStrategyAverager
from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.core import GameSpec
from liars_poker.policies.neural import compile_neural_to_dense
from liars_poker.serialization import save_policy

if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.set_float32_matmul_precision('high')
else:
    device = torch.device('cpu')
    print('WARNING: CUDA is not available; this notebook will run much slower on CPU.')

print('repo:', REPO_ROOT)
print('torch:', torch.__version__)
print('device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

# This is intentionally long: the point is to see whether suffix/epoch averages
# beat the global average over a serious run, not over a tiny screen.
TRAINING_MINUTES = 360
TRAVERSALS_PER_PLAYER = 1024
SEED = 7

# Dense-current observation is diagnostic overhead and is not counted against
# measured neural training time. 5 minutes keeps the wall-clock overhead sane
# while still giving 72 observation points over the overnight run.
AVERAGE_OBSERVE_EVERY_MINUTES = 5

# Exact evaluation is also diagnostic overhead and is not counted against measured
# neural training time. This schedule keeps the number of exact BR calls sane.
EVALUATION_TARGET_MINUTES = [5, 10, 15, 30, 45, 60, 90, 120, 180, 240, 300, 360]
CHECKPOINT_EVERY_MINUTES = 60
PRINT_EVERY_MINUTES = 5
LOG_EVERY_ITERATIONS = 1

# Suffix starts. These are aligned with the 5-minute observation cadence.
SUFFIX_START_MINUTES = [15, 30, 60, 120, 180, 240, 300]
GLOBAL_WEIGHTED_SUFFIX_START_MINUTES = [60, 180, 300]
EPOCH_WINDOW_MINUTES = [30, 60, 120]

CURRENT_POLICY_BATCH_SIZE = 65_536
EVAL_COMPILE_BATCH_SIZE = 65_536

TRAINER_KWARGS = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': SEED,
    'regret_buffer_capacity': 500_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'regret_train_steps': 24,
    'strategy_train_steps': 6,
    'regret_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native' if device.type == 'cuda' else 'recursive',
    'traversal_batch_size': 1024,
    'device_replay': device.type == 'cuda',
    'fused_optimizer': device.type == 'cuda',
    'amp_dtype': None,
    'compile_models': False,
}

run_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_DIR = (
    REPO_ROOT / 'artifacts' / 'cfr_plus_average_reset_exact_18_claim'
    / f'{spec.to_short_str()}___{run_id}'
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_JSONL = RUN_DIR / 'training.jsonl'
EVAL_JSONL = RUN_DIR / 'exact_suffix_results.jsonl'
MANIFEST_PATH = RUN_DIR / 'manifest.json'
CHECKPOINT_PATH = RUN_DIR / 'latest_checkpoint.pt'
TRACKER_STATE_PATH = RUN_DIR / 'exact_tracker_state.pkl'

manifest = {
    'run_type': 'cfr_plus_average_reset_snapshot_18_claim',
    'spec': spec.to_json(),
    'training_minutes': TRAINING_MINUTES,
    'traversals_per_player': TRAVERSALS_PER_PLAYER,
    'seed': SEED,
    'average_observe_every_minutes': AVERAGE_OBSERVE_EVERY_MINUTES,
    'evaluation_target_minutes': EVALUATION_TARGET_MINUTES,
    'suffix_start_minutes': SUFFIX_START_MINUTES,
    'global_weighted_suffix_start_minutes': GLOBAL_WEIGHTED_SUFFIX_START_MINUTES,
    'epoch_window_minutes': EPOCH_WINDOW_MINUTES,
    'trainer_kwargs': {**TRAINER_KWARGS, 'device': str(device)},
    'created_utc': datetime.now(timezone.utc).isoformat(),
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('spec:', spec.to_short_str())
print('run directory:', RUN_DIR)


In [ ]:
def append_jsonl(path: Path, row: dict) -> None:
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row) + '\n')


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]


def atomic_checkpoint(trainer: DeepCFRPlusTrainer, destination: Path) -> float:
    start = time.perf_counter()
    temporary = destination.with_suffix(destination.suffix + '.new')
    trainer.save_checkpoint(temporary)
    os.replace(temporary, destination)
    return time.perf_counter() - start


def exact_exploitability(policy) -> dict:
    start = time.perf_counter()
    _, meta = best_response_dense(
        spec,
        policy,
        debug=False,
        store_state_values=False,
    )
    p_first, p_second = meta['computer'].exploitability()
    predicted_avg = 0.5 * (p_first + p_second)
    return {
        'p_first': float(p_first),
        'p_second': float(p_second),
        'predicted_avg': float(predicted_avg),
        'exploitability': float(2.0 * predicted_avg - 1.0),
        'exact_br_s': float(time.perf_counter() - start),
    }


@dataclass
class AverageTracker:
    label: str
    kind: str
    weighting: str
    start_s: float = 0.0
    window_s: float | None = None

    def __post_init__(self):
        self.averager: ExactDenseStrategyAverager | None = None
        self.active = False
        self.observations = 0              # represented CFR+ iterations
        self.dense_observations = 0        # actual dense snapshots compiled
        self.start_iteration: int | None = None
        self.last_observed_iteration: int | None = None
        self.active_start_s: float | None = None
        self.epoch_index: int | None = None

    def reset(self, measured_s: float, iteration: int, *, start_s: float | None = None) -> None:
        self.averager = ExactDenseStrategyAverager(spec)
        self.active = True
        self.observations = 0
        self.dense_observations = 0
        self.start_iteration = int(iteration)
        self.last_observed_iteration = int(iteration)
        self.active_start_s = float(measured_s if start_s is None else start_s)

    def update(self, measured_s: float, iteration: int) -> None:
        if self.kind in {'global', 'suffix'}:
            if not self.active and measured_s >= self.start_s:
                self.reset(measured_s, iteration, start_s=self.start_s)
            return

        if self.kind == 'epoch':
            assert self.window_s is not None
            epoch_index = int(measured_s // self.window_s)
            if self.epoch_index != epoch_index:
                self.epoch_index = epoch_index
                self.reset(
                    measured_s,
                    iteration,
                    start_s=epoch_index * self.window_s,
                )
            return

        raise ValueError(f'Unknown tracker kind: {self.kind}')

    def observe(self, policy, trainer: DeepCFRPlusTrainer) -> None:
        if not self.active or self.averager is None:
            return
        current_iteration = int(trainer.iteration)
        previous_iteration = (
            current_iteration
            if self.last_observed_iteration is None
            else int(self.last_observed_iteration)
        )
        delta_iterations = max(0, current_iteration - previous_iteration)
        if delta_iterations <= 0:
            return

        if self.weighting == 'global_linear':
            # Approximate sum_{t=previous+1}^{current} t * sigma_current.
            weight = 0.5 * (
                (previous_iteration + 1) + current_iteration
            ) * delta_iterations
        elif self.weighting == 'local_linear':
            # Local iteration count starts at one after reset.
            start_local = self.observations + 1
            end_local = self.observations + delta_iterations
            weight = 0.5 * (start_local + end_local) * delta_iterations
        elif self.weighting == 'uniform':
            weight = float(delta_iterations)
        else:
            raise ValueError(f'Unknown weighting: {self.weighting}')

        self.averager.observe(policy, weight=float(weight))
        self.observations += int(delta_iterations)
        self.dense_observations += 1
        self.last_observed_iteration = current_iteration

    def metadata(self) -> dict:
        return {
            'label': self.label,
            'kind': self.kind,
            'weighting': self.weighting,
            'start_min': None if self.active_start_s is None else self.active_start_s / 60.0,
            'window_min': None if self.window_s is None else self.window_s / 60.0,
            'start_iteration': self.start_iteration,
            'observations': self.observations,
            'dense_observations': self.dense_observations,
            'active': self.active,
        }


def build_trackers() -> list[AverageTracker]:
    trackers = [
        AverageTracker(
            label='global_linear_from_0m',
            kind='global',
            weighting='global_linear',
            start_s=0.0,
        )
    ]
    for minute in SUFFIX_START_MINUTES:
        trackers.append(AverageTracker(
            label=f'suffix_local_linear_from_{minute}m',
            kind='suffix',
            weighting='local_linear',
            start_s=60.0 * minute,
        ))
        trackers.append(AverageTracker(
            label=f'suffix_uniform_from_{minute}m',
            kind='suffix',
            weighting='uniform',
            start_s=60.0 * minute,
        ))
    for minute in GLOBAL_WEIGHTED_SUFFIX_START_MINUTES:
        trackers.append(AverageTracker(
            label=f'suffix_global_linear_from_{minute}m',
            kind='suffix',
            weighting='global_linear',
            start_s=60.0 * minute,
        ))
    for window in EPOCH_WINDOW_MINUTES:
        trackers.append(AverageTracker(
            label=f'rolling_epoch_local_linear_{window}m',
            kind='epoch',
            weighting='local_linear',
            window_s=60.0 * window,
        ))
    return trackers


def evaluate_all(
    trainer: DeepCFRPlusTrainer,
    trackers: list[AverageTracker],
    *,
    target_min: float,
    measured_s: float,
    wall_s: float,
) -> list[dict]:
    rows = []

    learned_start = time.perf_counter()
    learned_dense = compile_neural_to_dense(
        trainer.average_policy(),
        batch_size=EVAL_COMPILE_BATCH_SIZE,
    )
    row = {
        'evaluation_target_min': float(target_min),
        'measured_training_s': float(measured_s),
        'wall_s': float(wall_s),
        'iteration': int(trainer.iteration),
        'label': 'learned_average_network',
        'kind': 'learned_network',
        'weighting': None,
        'start_min': None,
        'window_min': None,
        'start_iteration': None,
        'observations': None,
        'compile_s': float(time.perf_counter() - learned_start),
    }
    row.update(exact_exploitability(learned_dense))
    rows.append(row)
    del learned_dense

    for tracker in trackers:
        if not tracker.active or tracker.averager is None or tracker.observations == 0:
            continue
        compile_start = time.perf_counter()
        average_policy = tracker.averager.average_policy()
        row = {
            'evaluation_target_min': float(target_min),
            'measured_training_s': float(measured_s),
            'wall_s': float(wall_s),
            'iteration': int(trainer.iteration),
            **tracker.metadata(),
            'compile_s': float(time.perf_counter() - compile_start),
        }
        row.update(exact_exploitability(average_policy))
        rows.append(row)
        del average_policy

    for row in rows:
        append_jsonl(EVAL_JSONL, row)

    return rows


def save_tracker_state(trackers: list[AverageTracker], path: Path) -> None:
    # Exact averagers are numpy arrays and pickle cleanly on this small spec.
    temporary = path.with_suffix(path.suffix + '.new')
    with temporary.open('wb') as f:
        pickle.dump(trackers, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temporary, path)


In [ ]:
trainer = DeepCFRPlusTrainer(spec, **TRAINER_KWARGS)
trackers = build_trackers()
for tracker in trackers:
    tracker.update(0.0, 0)

training_budget_s = 60.0 * TRAINING_MINUTES
evaluation_targets_s = [60.0 * minute for minute in EVALUATION_TARGET_MINUTES]
next_eval_index = 0
next_checkpoint_s = 60.0 * CHECKPOINT_EVERY_MINUTES
next_print_s = 0.0
next_average_observe_s = 60.0 * AVERAGE_OBSERVE_EVERY_MINUTES

measured_s = 0.0
wall_start = time.perf_counter()

print('Starting six-hour measured CFR+ run.')
print('Run directory:', RUN_DIR)

while measured_s < training_budget_s:
    iteration_start = time.perf_counter()
    record = trainer.run_iteration(traversals_per_player=TRAVERSALS_PER_PLAYER)
    iteration_s = time.perf_counter() - iteration_start
    measured_s += iteration_s

    exact_observe_s = 0.0
    did_exact_observe = False
    while measured_s >= next_average_observe_s:
        observe_start = time.perf_counter()
        current_dense = trainer.current_policy_dense(batch_size=CURRENT_POLICY_BATCH_SIZE)
        for tracker in trackers:
            tracker.update(next_average_observe_s, trainer.iteration)
            tracker.observe(current_dense, trainer)
        exact_observe_s += time.perf_counter() - observe_start
        did_exact_observe = True
        del current_dense
        next_average_observe_s += 60.0 * AVERAGE_OBSERVE_EVERY_MINUTES
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    wall_s = time.perf_counter() - wall_start
    timing = record.get('timing', {})
    compact_record = {
        'iteration': int(trainer.iteration),
        'measured_training_s': float(measured_s),
        'wall_s': float(wall_s),
        'iteration_s': float(iteration_s),
        'exact_observe_s': float(exact_observe_s),
        'did_exact_observe': bool(did_exact_observe),
        'traversal_s': float(timing.get('traversal_s', np.nan)),
        'regret_training_s': float(timing.get('regret_training_s', np.nan)),
        'strategy_training_s': float(timing.get('strategy_training_s', np.nan)),
        'regret_loss': record.get('regret_loss'),
        'strategy_loss': record.get('strategy_loss'),
        'regret_buffer_sizes': record.get('regret_buffer_sizes'),
        'strategy_buffer_sizes': record.get('strategy_buffer_sizes'),
        'regret_records_seen': record.get('regret_records_seen'),
        'strategy_records_seen': record.get('strategy_records_seen'),
        'active_exact_trackers': sum(
            1 for tracker in trackers if tracker.active and tracker.observations > 0
        ),
    }
    if trainer.validation_fraction > 0.0 and trainer.iteration % 25 == 0:
        validation = trainer.validation_metrics()
        compact_record['validation'] = validation
    if trainer.iteration % LOG_EVERY_ITERATIONS == 0:
        append_jsonl(TRAINING_JSONL, compact_record)

    while (
        next_eval_index < len(evaluation_targets_s)
        and measured_s >= evaluation_targets_s[next_eval_index]
    ):
        target_min = EVALUATION_TARGET_MINUTES[next_eval_index]
        print(f'\n[evaluate] target={target_min}m actual={measured_s/60:.2f}m iter={trainer.iteration}')
        eval_rows = evaluate_all(
            trainer,
            trackers,
            target_min=target_min,
            measured_s=measured_s,
            wall_s=time.perf_counter() - wall_start,
        )
        eval_frame = pd.DataFrame(eval_rows)
        display(
            eval_frame[['label', 'kind', 'weighting', 'start_min', 'window_min', 'observations', 'dense_observations', 'exploitability']]
            .sort_values('exploitability')
            .head(12)
            .style.format(precision=6)
        )
        save_tracker_state(trackers, TRACKER_STATE_PATH)
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        next_eval_index += 1

    if measured_s >= next_checkpoint_s:
        checkpoint_s = atomic_checkpoint(trainer, CHECKPOINT_PATH)
        save_tracker_state(trackers, TRACKER_STATE_PATH)
        print(
            f'[checkpoint] train={measured_s/60:.1f}m iter={trainer.iteration} '
            f'time={checkpoint_s:.1f}s'
        )
        next_checkpoint_s += 60.0 * CHECKPOINT_EVERY_MINUTES

    if measured_s >= next_print_s:
        print(
            f"[train] {measured_s/60:7.2f}m / {TRAINING_MINUTES}m "
            f"iter={trainer.iteration:7d} "
            f"iter_s={iteration_s:.3f} observe_s={exact_observe_s:.3f} "
            f"active_avg={compact_record['active_exact_trackers']}"
        )
        next_print_s += 60.0 * PRINT_EVERY_MINUTES

final_checkpoint_s = atomic_checkpoint(trainer, CHECKPOINT_PATH)
save_tracker_state(trackers, TRACKER_STATE_PATH)

FINAL_POLICY_DIR = RUN_DIR / 'final_learned_average_policy'
save_policy(trainer.average_policy(), str(FINAL_POLICY_DIR))

EXACT_POLICY_ROOT = RUN_DIR / 'final_exact_average_policies'
EXACT_POLICY_ROOT.mkdir(exist_ok=True)
for tracker in trackers:
    if tracker.active and tracker.averager is not None and tracker.observations > 0:
        save_policy(
            tracker.averager.average_policy(),
            str(EXACT_POLICY_ROOT / tracker.label),
        )

manifest['status'] = 'complete'
manifest['completed_utc'] = datetime.now(timezone.utc).isoformat()
manifest['final_iteration'] = int(trainer.iteration)
manifest['final_measured_training_s'] = float(measured_s)
manifest['final_wall_s'] = float(time.perf_counter() - wall_start)
manifest['final_checkpoint_s'] = float(final_checkpoint_s)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('Completed run:', RUN_DIR)
print('Final measured training minutes:', measured_s / 60.0)


In [ ]:
training = pd.DataFrame(read_jsonl(TRAINING_JSONL))
evaluations = pd.DataFrame(read_jsonl(EVAL_JSONL))

print('training rows:', len(training))
print('evaluation rows:', len(evaluations))
display(evaluations.tail(10).style.format(precision=6))


In [ ]:
if evaluations.empty:
    raise RuntimeError('No evaluation rows found.')

latest_target = evaluations['evaluation_target_min'].max()
latest = evaluations[evaluations['evaluation_target_min'] == latest_target].copy()

summary_cols = [
    'label',
    'kind',
    'weighting',
    'start_min',
    'window_min',
    'observations',
    'iteration',
    'exploitability',
    'p_first',
    'p_second',
]

latest_summary = latest[summary_cols].sort_values('exploitability').reset_index(drop=True)
display(latest_summary.style.format(precision=6))
latest_summary.to_csv(RUN_DIR / 'final_exact_suffix_summary.csv', index=False)

best_by_label = (
    evaluations.sort_values('exploitability')
    .groupby('label', as_index=False)
    .first()
    .sort_values('exploitability')
)
display(best_by_label[summary_cols + ['evaluation_target_min']].head(20).style.format(precision=6))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 11))

selected_labels = [
    'learned_average_network',
    'global_linear_from_0m',
    'suffix_local_linear_from_30m',
    'suffix_local_linear_from_60m',
    'suffix_local_linear_from_120m',
    'suffix_local_linear_from_180m',
    'suffix_local_linear_from_240m',
    'suffix_local_linear_from_300m',
]

for label in selected_labels:
    frame = evaluations[evaluations['label'] == label].sort_values('measured_training_s')
    if frame.empty:
        continue
    axes[0, 0].plot(
        frame['measured_training_s'] / 60.0,
        frame['exploitability'],
        marker='o',
        label=label,
    )
axes[0, 0].set(
    title='Global versus fixed suffix exact averages',
    xlabel='Measured CFR+ training minutes',
    ylabel='Exact exploitability',
)
axes[0, 0].set_yscale('log')
axes[0, 0].grid(True, which='both', alpha=0.3)
axes[0, 0].legend(fontsize=8)

suffix_final = latest[
    latest['kind'].eq('suffix')
    & latest['weighting'].isin(['local_linear', 'uniform', 'global_linear'])
].copy()
for weighting, frame in suffix_final.groupby('weighting'):
    frame = frame.sort_values('start_min')
    axes[0, 1].plot(
        frame['start_min'],
        frame['exploitability'],
        marker='o',
        label=weighting,
    )
axes[0, 1].axhline(
    latest.loc[latest['label'].eq('global_linear_from_0m'), 'exploitability'].iloc[0],
    color='black',
    linestyle='--',
    linewidth=1,
    label='global from 0m',
)
axes[0, 1].set(
    title=f'Final suffix quality at {latest_target:.0f}m',
    xlabel='Suffix start minute',
    ylabel='Exact exploitability',
)
axes[0, 1].set_yscale('log')
axes[0, 1].grid(True, which='both', alpha=0.3)
axes[0, 1].legend(fontsize=8)

epoch_rows = evaluations[evaluations['kind'].eq('epoch')].copy()
for label, frame in epoch_rows.groupby('label'):
    frame = frame.sort_values('measured_training_s')
    axes[1, 0].plot(
        frame['measured_training_s'] / 60.0,
        frame['exploitability'],
        marker='o',
        label=label,
    )
axes[1, 0].set(
    title='Rolling epoch exact averages',
    xlabel='Measured CFR+ training minutes',
    ylabel='Exact exploitability',
)
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, which='both', alpha=0.3)
axes[1, 0].legend(fontsize=8)

if not training.empty:
    axes[1, 1].plot(
        training['measured_training_s'] / 60.0,
        training['iteration_s'],
        alpha=0.7,
        label='trainer iteration',
    )
    axes[1, 1].plot(
        training['measured_training_s'] / 60.0,
        training['exact_observe_s'],
        alpha=0.7,
        label='exact-average observation overhead',
    )
    axes[1, 1].set(
        title='Per-iteration timing',
        xlabel='Measured CFR+ training minutes',
        ylabel='Seconds',
    )
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend(fontsize=8)

fig.tight_layout()
plt.show()


In [ ]:
# Compact interpretation helper.
global_final = latest.loc[
    latest['label'].eq('global_linear_from_0m'),
    'exploitability',
].iloc[0]

learned_final = latest.loc[
    latest['label'].eq('learned_average_network'),
    'exploitability',
].iloc[0]

suffix_candidates = latest[latest['kind'].isin(['suffix', 'epoch'])].copy()
best_suffix = suffix_candidates.sort_values('exploitability').iloc[0]

diagnostic_summary = pd.DataFrame([{
    'final learned average': learned_final,
    'final global exact average': global_final,
    'best final suffix/epoch exact average': best_suffix['exploitability'],
    'best final suffix/epoch label': best_suffix['label'],
    'best suffix improvement over global': global_final - best_suffix['exploitability'],
    'best suffix improvement fraction': (
        (global_final - best_suffix['exploitability']) / global_final
        if global_final else np.nan
    ),
}])

display(diagnostic_summary.style.format(precision=6))
diagnostic_summary.to_csv(RUN_DIR / 'diagnostic_summary.csv', index=False)


## How to read the result

- If a suffix or rolling epoch snapshot average clearly beats the global snapshot average, then average inertia is real.
- If all suffix/epoch averages are worse than the global average, the historical average is helping and reset averaging is unlikely to solve the 69-claim plateau by itself.
- If medium windows win but short windows lose, use epoch/window averaging rather than only the latest strategy.
- If the learned average network is much worse than the exact global/suffix snapshot averages, then the average-network fit is a separate bottleneck.
- The snapshot averages are weighted by represented CFR+ iterations, but they are not exact every-iteration averages. This is deliberate to keep the six-hour run practical.
